# Сбор данных котировок и новостей

Источники: **MOEX ISS** (котировки, часовые свечи), **Интерфакс** (новости).

## 1. Настройка

Импорты, период собираемых данных и папки для файлов.

In [1]:
from pathlib import Path
from urllib.parse import urljoin
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

DATE_START = "2020-01-01"
DATE_END = "2026-10-23"  # последний день с торгами в выбранном периоде
TZ = "Europe/Moscow"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

STOCKS_DIR = DATA_DIR / "stocks_by_ticker"
STOCKS_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}

## 2. Котировки MOEX

44 акции. Выбраны  интересующие меня скучные базированые сектора: нефтегаз, металлы, добыча, удобрения и энергетика + акции текущего индекса Мосбиржи

Получаем часовые свечи из режима торгов `TQBR` через ISS API. 

In [2]:
TICKERS = [
    'LKOH', 'SBER', 'GAZP', 'YDEX', 'TATN', 'T', 'GMKN', 'NVTK',
    'ROSN', 'X5', 'VTBR', 'SBERP', 'OZON', 'PLZL', 'SNGSP', 'SNGS',
    'HEAD', 'TATNP', 'MOEX', 'MTSS', 'IRAO', 'RUAL', 'CBOM', 'CHMF',
    'NLMK', 'PHOR', 'DOMRF', 'SVCB', 'MAGN', 'TRNFP', 'RTKM', 'ENPG',
    'AFLT', 'MDMG', 'BSPB', 'FLOT', 'ALRS', 'CNRU', 'POSI', 'UGLD',
    'AFKS', 'RAGR', 'VKCO', 'RENI',
]

CANDLE_INTERVAL = 60  # минут
BOARD = "TQBR"

### 2.1. Функция загрузки

MOEX отдаёт данные кусками: параметр `start` сдвигает начало следующей страницы.

In [3]:
def get_candles(ticker):
    url = (
        "https://iss.moex.com/iss/engines/stock/markets/shares/"
        f"boards/{BOARD}/securities/{ticker}/candles.json"
    )
    params = {
        "from": DATE_START,
        "till": DATE_END,
        "interval": CANDLE_INTERVAL,
        "start": 0,
        "iss.meta": "off",
    }
    all_rows = []
    columns = None

    while True:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        block = response.json()["candles"]
        columns, rows = block["columns"], block["data"]
        if not rows:
            break
        all_rows.extend(rows)
        params["start"] += len(rows)
        time.sleep(0.3)

    df = pd.DataFrame(all_rows, columns=columns)
    if not df.empty:
        df = df.drop_duplicates("begin").sort_values("begin").reset_index(drop=True)
    df["ticker"] = ticker
    return df

### 2.2. Выгрузка и объединение

Каждый тикер сохраняем отдельно. При повторном запуске используем локальный файл.

In [4]:
stock_frames = []
stock_errors = []

for ticker in TICKERS:
    path = STOCKS_DIR / f"{ticker}_{DATE_START}_{DATE_END}_1h.csv"
    try:
        if path.exists():
            df = pd.read_csv(path)
        else:
            df = get_candles(ticker)
            if df.empty:
                raise ValueError("Нет свечей")
            df.to_csv(path, index=False, encoding="utf-8-sig")

        df["ticker"] = ticker
        stock_frames.append(df)
        print(f"{ticker}: {len(df)} свечей")
    except (requests.RequestException, ValueError, KeyError) as exc:
        stock_errors.append((ticker, str(exc)))
        print(f"Ошибка {ticker}: {exc}")

if not stock_frames:
    raise RuntimeError("Не удалось получить ни одной акции")

stocks = (
    pd.concat(stock_frames, ignore_index=True)
    .drop_duplicates(["ticker", "begin"])
    .sort_values(["ticker", "begin"])
    .reset_index(drop=True)
)
stocks.to_csv(DATA_DIR / "stocks.csv", index=False, encoding="utf-8-sig")

print(f"Итого: {stocks['ticker'].nunique()} акций, {len(stocks)} свечей")
print("Ошибки:", stock_errors)

LKOH: 27242 свечей
SBER: 27245 свечей
GAZP: 27248 свечей
YDEX: 11213 свечей
TATN: 27216 свечей
T: 26639 свечей
GMKN: 27183 свечей
NVTK: 27201 свечей
ROSN: 27241 свечей
X5: 9432 свечей
VTBR: 27180 свечей
SBERP: 27239 свечей
OZON: 22749 свечей
PLZL: 27128 свечей
SNGSP: 27222 свечей
SNGS: 27195 свечей
HEAD: 10235 свечей
TATNP: 27195 свечей
MOEX: 27236 свечей
MTSS: 27213 свечей
IRAO: 27220 свечей
RUAL: 27040 свечей
CBOM: 26825 свечей
CHMF: 27244 свечей
NLMK: 27236 свечей
PHOR: 26994 свечей
DOMRF: 4739 свечей
SVCB: 13275 свечей
MAGN: 27102 свечей
TRNFP: 26692 свечей
RTKM: 27050 свечей
ENPG: 24604 свечей
AFLT: 27243 свечей
MDMG: 19033 свечей
BSPB: 23305 свечей
FLOT: 22372 свечей
ALRS: 27244 свечей
CNRU: 7852 свечей
POSI: 19681 свечей
UGLD: 12241 свечей
AFKS: 27067 свечей
RAGR: 8637 свечей
VKCO: 20061 свечей
RENI: 16804 свечей
Итого: 44 акций, 1006013 свечей
Ошибки: []


### 2.3. Техническая проверка

Сверяем диапазоны дат, дубликаты и пустые значения. Допускаем разное количество свеч

In [5]:
stock_check = stocks.groupby("ticker").agg(
    candles=("begin", "size"),
    first=("begin", "min"),
    last=("begin", "max"),
)
display(stock_check.sort_values("candles"))
print("Дубликаты:", stocks.duplicated(["ticker", "begin"]).sum())
print("Пустые значения:", stocks.isna().sum().sum())

,candles,first,last
ticker,,,
DOMRF,4739,2025-11-20 16:00:00,2026-10-23 23:00:00
CNRU,7852,2025-04-03 09:00:00,2026-10-23 23:00:00
RAGR,8637,2025-02-17 09:00:00,2026-10-23 23:00:00
X5,9432,2025-01-09 09:00:00,2026-10-23 23:00:00
HEAD,10235,2024-09-26 09:00:00,2026-10-23 23:00:00
YDEX,11213,2024-07-24 09:00:00,2026-10-23 23:00:00
UGLD,12241,2023-11-22 16:00:00,2026-10-23 23:00:00
SVCB,13275,2023-12-15 17:00:00,2026-10-23 23:00:00
RENI,16804,2021-10-21 09:00:00,2026-10-23 23:00:00


Дубликаты: 0
Пустые значения: 0


## 3. Интерфакс

У Интерфакса есть архив новостей, можно найти по дате. Ищем.

Парсим архив по дням: `datetime`, `title`, `url`, `source`. 

### 3.1. Парсер одного дня

пример HTML-элемента из новостного архива:

```html
<div class="news-item">
    <span class="time">11:34</span>
    <a href="/business/1054414">
        Текст заголовка новости
    </a>
</div>
```

Основные элементы следующие:

- `<div>` — контейнер, объединяющий элементы новости.
- `<span>` — элемент, содержащий время публикации.
- `<a>` — ссылка на новость.
- `href` — атрибут с адресом статьи.
- `11:34` — время публикации.
- `Текст заголовка новости` — заголовок.


Собственно методом ниже находим ссылки на публикации и время в HTML-блоке вокруг заголовка.

In [6]:
def get_interfax_news(date):
    date = pd.Timestamp(date)
    archive_url = f"https://www.interfax.ru/business/news/{date:%Y/%m/%d}/"
    response = requests.get(archive_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    records = []
    for a in soup.find_all("a", href=True):
        link = urljoin(archive_url, a["href"])

        # проверяем соответствие ссылки формату статьи интерфакса
        if not re.fullmatch(r"https://www\.interfax\.ru/[a-z-]+/\d+/?", link):
            continue

        # проверяем наличие родительского элемента
        if a.parent is None:
            continue

        # ищем время публикации в родительском элементе. Если нет, пропускаем
        match = re.match(r"^(\d{1,2}:\d{2})\s+", a.parent.get_text(" ", strip=True))
        if not match:
            continue
        
        title = a.get_text(" ", strip=True)
        if title:
            records.append({
                "datetime": f"{date:%Y-%m-%d} {match.group(1)}",
                "title": title,
                "url": link,
                "source": "Interfax",
            })

    return pd.DataFrame(
        records, columns=["datetime", "title", "url", "source"]
    ).drop_duplicates("url")

### 3.2. Полный архив за выбранный период

Сохраняем каждый день отдельно; неудачные даты можно загрузить повторным запуском.

In [7]:
INTERFAX_DIR = DATA_DIR / "interfax_daily"
INTERFAX_DIR.mkdir(exist_ok=True)

interfax_errors = []
for date in pd.date_range(DATE_START, DATE_END, freq="D"):
    day = date.strftime("%Y-%m-%d")
    path = INTERFAX_DIR / f"{day}.csv"
    if path.exists():
        continue
    try:
        df_day = get_interfax_news(day)
        df_day.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"{day}: {len(df_day)}")
    except requests.RequestException as exc:
        interfax_errors.append((day, str(exc)))
        print(f"Ошибка {day}: {exc}")
    time.sleep(1.5)

print("Ошибок:", len(interfax_errors))

Ошибок: 0


Проверим, что по всем дням были собраны новости:

In [8]:
archive_dir = DATA_DIR / "interfax_daily"

files = sorted(archive_dir.glob("*.csv"))

expected_days = {
    d.strftime("%Y-%m-%d")
    for d in pd.date_range(DATE_START, DATE_END)
}

downloaded_days = {p.stem for p in files}

missing_days = sorted(expected_days - downloaded_days)

print("Ожидалось дней:", len(expected_days))
print("Есть дневных файлов:", len(files))
print("Отсутствуют дни:", len(missing_days))
print("Первые отсутствующие:", missing_days[:10])

print("Пустых файлов (0 байт):",
      sum(p.stat().st_size == 0 for p in files))

Ожидалось дней: 2488
Есть дневных файлов: 2488
Отсутствуют дни: 0
Первые отсутствующие: []
Пустых файлов (0 байт): 0


### 3.3. Общий CSV

Объединяем дневные файлы в один большой csv.

In [9]:
interfax_files = sorted(INTERFAX_DIR.glob("*.csv"))

# проверка что есть все дневные файлы, все дни загружены
assert len(interfax_files) == len(pd.date_range(DATE_START, DATE_END)), (
    "Не все дни загружены"
)

news = pd.concat(
    [pd.read_csv(path) for path in interfax_files], ignore_index=True
)
news = news.drop_duplicates("url")
news["datetime"] = pd.to_datetime(news["datetime"])
news = news.sort_values("datetime").reset_index(drop=True)
news.to_csv(DATA_DIR / "interfax_news.csv", index=False, encoding="utf-8-sig")

print(f"Дней: {len(interfax_files)}; новостей: {len(news)}")
print("Диапазон:", news["datetime"].min(), "—", news["datetime"].max())

Дней: 2488; новостей: 106006
Диапазон: 2020-01-01 06:42:00 — 2026-10-23 22:31:00


### 3.4. Попытка в фильтрацию по темам

Очень грубо (с применением тематических слов и регулярок) пытаемся отфильтровать только нужные новости

In [10]:
PATTERNS = {
    "oil_gas": [
        r"\b(?:газпром|лукойл|роснефт|новатэк|татнефт|сургутнефт)\w*",
        r"\bнефт\w*",
        r"\bгаз(?:а|у|ом|е|ов\w*|опровод\w*)\b",
        r"\b(?:опек|brent|нпз)\b",
        r"\bуглеводород\w*",
        r"\bсахалинск\w*\s+энерги\w*",
    ],
    "metals": [
        r"\b(?:норникел|нлмк|северстал|ммк|русал)\w*",
        r"\b(?:металлург|алюмин|никел)\w*",
        r"\b(?:сталь|стали|стальн\w*)\b",
        r"\b(?:медь|меди|медн\w*)\b",
    ],
    "mining": [
        r"\b(?:полюс|селигдар|алрос)\w*",
        r"\b(?:золот|алмаз|паллади|платин)\w*",
    ],
    "fertilizers": [
        r"\b(?:фосагро|акрон|удобрени|фосфат|калийн|калий)\w*",
    ],
    "energy": [
        r"\b(?:интер\s+рао|русгидро|юнипро)\b",
        r"\b(?:электроэнерг|электростанц|электросет)\w*",
        r"\b(?:энергосистем|энергоснабж|энерготариф)\w*",
    ],
    "finance": [
        r"\b(?:сбербанк|сбер|втб|тинькофф|т[ -]?банк|т[ -]?технологи|мосбирж|московск\w*\s+бирж\w*)\w*",
        r"\b(?:совкомбанк|московск\w*\s+кредитн\w*\s+банк\w*|мкб|банк\w*\s+санкт[- ]петербург|бспб)\b",
        r"\b(?:дом[. -]?рф|дом\.рф|ренессанс\s+страховани\w*)\b",
    ],
    "it": [
        r"\b(?:яндекс|yandex|хэдхантер|headhunter|вконтакте|групп\w*\s+вк|vkco)\b",
        r"\b(?:позитив\s+текнолоджиз|positive\s+technologies|групп\w*\s+позитив|циан|cian)\b",
    ],
    "retail": [
        r"\b(?:озон|ozon|икс\s*5|x5|пят[её]рочк\w*)\b",
    ],
    "telecom": [
        r"\b(?:мтс|ростелеком|мобильн\w*\s+телесистем\w*)\b",
    ],
    "transport": [
        r"\b(?:аэрофлот|совкомфлот|транснефт)\w*",
    ],
    "healthcare": [
        r"\b(?:мать\s+и\s+дитя|мд\s+медикал|md\s+medical)\b",
    ],
    "agriculture": [
        r"\b(?:русагро|групп\w*\s+русагро)\w*",
    ],
    "holdings": [
        r"\bафк\s*[«\"]?система\b",
    ],
    "macro": [
        r"\bключев\w*\s+ставк\w*",
        r"\b(?:цб|банк\w*\s+россии)\b.*\b(?:ставк|дкп|дку|инфляц|проинфляц|денежно-кредитн)\w*",
        r"\bставк\w*.*\bцб\b",
        r"\bинфляц\w*.*\b(?:рф|росси\w*)\b",
        r"\b(?:рф|росси\w*)\b.*\bинфляц\w*",
        r"\bсанкц\w*",
        r"\bмеждународн\w*\s+резерв\w*",
        r"\bденежн\w*\s+масс\w*",
        r"\bрубл\w*.*\b(?:юан\w*|доллар\w*|евро|курс\w*|укреп\w*|ослаб\w*|сниж\w*|подешев\w*)",
        r"\b(?:курс\w*|юан\w*|доллар\w*|евро).*?\bрубл\w*",
    ],
}

# Дополнительные варианты названий компаний, выявленные при проверке полноты выборки
PATTERNS["metals"].extend([
    r"\b(?:эн\s*\+\s*групп\w*|эн\s+плюс|en\+\s*group)\b",
])
PATTERNS["mining"].extend([
    r"\b(?:югк|южуралзолот\w*|южн\w*\s+урал\w*\s+золот\w*)\b",
])

# Дополнения после аудита пропущенных публикаций
PATTERNS["oil_gas"].extend([
    r"\blukoil(?:-\w+)?\b",
])

PATTERNS["finance"].extend([
    r"\bсовкомбанк\w*\b",
    r'\bткс\s+[«"]?холдинг\w*\b',
    r"\bsber\b",
])

PATTERNS["it"].extend([
    r"\bяндекс\w*\b",
    r"\b(?:vk|вк)\b",
    r"\bmail\.ru(?:\s+group)?\b",
    r"\bхэдхантер\w*\b",
])

PATTERNS["retail"].extend([
    r"(?<!\w)[xх]5(?!\w)",
    r"\bперекр[её]ст(?:ок|к\w*)\b",
])

PATTERNS["telecom"].extend([
    r"\bростелеком\w*\b",
])

PATTERNS["metals"].extend([
    r"(?<!\w)(?:эн\s*\+|en\s*\+\s*group)(?!\w)",
])

PATTERNS["finance"].append(
    r'\b(?:бспб|bspb)\b|\bбанк\w*\s*[«"„]?\s*санкт[-\s]*петербург\w*\b'
)

COMPILED_PATTERNS = {
    topic: [re.compile(pattern, re.I) for pattern in rules]
    for topic, rules in PATTERNS.items()
}

def classify_news(title):
    title = str(title)
    matches = []

    for topic, rules in COMPILED_PATTERNS.items():
        if any(rule.search(title) for rule in rules):
            matches.append(topic)

    return ", ".join(matches)

interfax_filtered = news.copy()
interfax_filtered["topics"] = interfax_filtered["title"].apply(classify_news)
interfax_filtered = interfax_filtered[interfax_filtered["topics"] != ""].copy()
interfax_filtered.to_csv(
    DATA_DIR / "interfax_filtered.csv", index=False, encoding="utf-8-sig"
)
print(f"Отобрано: {len(interfax_filtered)} из {len(news)}")
display(interfax_filtered[["title", "topics"]].head(5))

Отобрано: 35525 из 106006


,title,topics
8,Exxon Mobil добилась в суде отмены штрафов в $...,oil_gas
9,Добыча нефти в России в 2019 году повысилась н...,oil_gas
11,"""Газпром"" конкретизировал итоговые данные по д...",oil_gas
13,Экспорт нефти из России в дальнее зарубежье в ...,oil_gas
14,Нефть резко дорожала на новостях об убийстве и...,oil_gas


В результат ячейки сверху это не попадет, но во время игр с новостями было обнаружено, что могут в фильтры попадаться невалидные новости (например про золотой купол залетала в тематику золота). Оставляем на потом.

### 3.5. Полные тексты статей

Этот этап добавлен после выгрузки заголовков. Тексты сохраняются в `data/interfax_with_text.csv`; загрузку можно продолжать после прерывания.

In [12]:
def get_interfax_text(url):
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    article = soup.find("article")

    if article is None:
        return None

    texts = []
    for p in article.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            texts.append(text)

    return "\n".join(texts) if texts else None


In [13]:
TEXT_FILE = DATA_DIR / "interfax_with_text.csv"
TEXT_DAILY_DIR = DATA_DIR / "interfax_text_daily"
TEXT_DAILY_DIR.mkdir(parents=True, exist_ok=True)

# Берём уже отфильтрованные новости
articles = interfax_filtered.copy()

# Проверяем ранее сохранённые статьи
if TEXT_FILE.exists():
    saved = pd.read_csv(TEXT_FILE)
    done = set(saved["url"])
else:
    done = set()

errors = []

for i, row in enumerate(articles.itertuples(), start=1):

    if row.url in done:
        continue

    try:
        text = get_interfax_text(row.url)

    except requests.HTTPError as e:
        status = e.response.status_code

        if status in (403, 429):
            print(f"Сервер ограничил доступ: HTTP {status}")
            break

        errors.append((row.url, str(e)))
        continue

    except requests.RequestException as e:
        errors.append((row.url, str(e)))
        continue

    record = {
        "datetime": row.datetime,
        "title": row.title,
        "url": row.url,
        "source": row.source,
        "topics": row.topics,
        "text": text,
        "text_status": "ok" if text else "empty",
        "retrieved_at_utc": pd.Timestamp.now(
            tz="UTC"
        ).isoformat()
    }

    record_df = pd.DataFrame([record])

    # Дописываем одну запись в CSV
    record_df.to_csv(
        TEXT_FILE,
        mode="a",
        header=not TEXT_FILE.exists(),
        index=False,
        encoding="utf-8"
    )

    # В тот же момент сохраняем статью в CSV её дня
    day = str(row.datetime)[:10]
    day_path = TEXT_DAILY_DIR / f"{day}.csv"
    record_df.to_csv(
        day_path,
        mode="a",
        header=not day_path.exists(),
        index=False,
        encoding="utf-8",
    )

    done.add(row.url)

    print(f"[{i}/{len(articles)}] {row.title[:55]}")

    time.sleep(2)

print("\nОбработка завершена")
print("Ошибок:", len(errors))
print("Обработано:", len(done))



[4014/35525] Банк "Санкт-Петербург" увеличил годовую прибыль по МСФО
[5723/35525] Банк "Санкт-Петербург" готов потратить 1,05 млрд рублей
[5957/35525] Квартальная прибыль банка "Санкт-Петербург" выросла в 2
[8965/35525] Банк "Санкт-Петербург" остановил выпуск карт UnionPay и
[9425/35525] Банк "Санкт-Петербург" отказался от моратория на банкро
[10558/35525] Банк "Санкт-Петербург" выплатит дивиденды только на "пр
[11278/35525] Акции банка "Санкт-Петербург" подорожали на 20% на ново
[11801/35525] Акции банка "Санкт-Петербург" раллируют на новостях о д
[12410/35525] Банк "Санкт-Петербург" подал иск к НРД и Euroclear почт
[12986/35525] Банк "Санкт-Петербург" решил выкупить 22 млн акций
[14380/35525] Суд удовлетворил иск банка "Санкт-Петербург" к Euroclea
[15274/35525] Чистая прибыль банка "Санкт-Петербург" по МСФО в 2022 г
[15846/35525] Акционеры банка "Санкт-Петербург" утвердили выплату див
[16100/35525] Банк "Санкт-Петербург" в I квартале получил 14,8 млрд р
[16106/35525] Банк "Санкт-Пете

In [14]:
# Восстанавливаем/обновляем дневные файлы из уже скачанных текстов.
# Можно запускать после остановки сбора, ничего повторно не скачивается.
if TEXT_FILE.exists():
    texts_by_day = pd.read_csv(TEXT_FILE)
    texts_by_day["_date"] = texts_by_day["datetime"].astype(str).str[:10]

    for day, df_day in texts_by_day.groupby("_date"):
        df_day.drop(columns="_date").to_csv(
            TEXT_DAILY_DIR / f"{day}.csv",
            index=False,
            encoding="utf-8-sig",
        )

    print(f"Ежедневных файлов с текстами: {texts_by_day['_date'].nunique()}")
else:
    print("Пока нет скачанных текстов")


Ежедневных файлов с текстами: 2120


In [15]:
articles = pd.read_csv(DATA_DIR / "interfax_with_text.csv")

expected_urls = set(interfax_filtered["url"])
downloaded_urls = set(articles["url"])

print("Ожидалось статей:", len(expected_urls))
print("Скачано статей:", len(articles))
print("Не скачано:", len(expected_urls - downloaded_urls))
print("Дубликатов:", articles["url"].duplicated().sum())
print("Пустых текстов:", articles["text"].isna().sum())

print("\nСтатусы:")
print(articles["text_status"].value_counts())

print("\nОчень коротких текстов (<100 символов):")
print(
    (articles["text"].fillna("").str.len() < 100).sum()
)

Ожидалось статей: 35525
Скачано статей: 35525
Не скачано: 0
Дубликатов: 0
Пустых текстов: 0

Статусы:
text_status
ok    35525
Name: count, dtype: int64

Очень коротких текстов (<100 символов):
0


## 4. Итоговая таблица новостей


In [16]:
interfax = interfax_filtered.merge(
    articles[["url", "text", "text_status", "retrieved_at_utc"]],
    on="url",
    how="left",
    validate="one_to_one",
)

assert interfax["text"].notna().all(), "Не для всех новостей скачан текст"
assert (interfax["text_status"] == "ok").all(), "Есть статьи с ошибочным статусом"

interfax["datetime"] = pd.to_datetime(interfax["datetime"]).dt.tz_localize(TZ)
interfax["record_id"] = "interfax_" + interfax["url"].astype(str)
interfax["event_type"] = "news"
interfax["time_source"] = "interfax_archive"

FINAL_COLUMNS = [
    "record_id", "datetime", "title", "url", "source",
    "topics", "event_type", "time_source", "text",
    "text_status", "retrieved_at_utc",
]

news_final = (
    interfax[FINAL_COLUMNS]
    .drop_duplicates("record_id")
    .sort_values("datetime")
    .reset_index(drop=True)
)
news_final.to_csv(DATA_DIR / "news_final.csv", index=False, encoding="utf-8-sig")


### 4.1. Примитивная проверка данных

Проверяем число записей, уникальность ID и диапазон дат. 

In [17]:
# print("Котировки:", len(stocks), "строк,", stocks["ticker"].nunique(), "тикеров")
print("Все новости Интерфакса:", len(news))
print("Отбор Интерфакса:", len(interfax_filtered))
print("Итоговых новостей:", len(news_final))
print("Дубликаты ID:", news_final["record_id"].duplicated().sum())
print("Пустые даты:", news_final["datetime"].isna().sum())
print("Диапазон:", news_final["datetime"].min(), "—", news_final["datetime"].max())
print("По источникам:")
print(news_final["source"].value_counts())

Все новости Интерфакса: 106006
Отбор Интерфакса: 35525
Итоговых новостей: 35525
Дубликаты ID: 0
Пустые даты: 0
Диапазон: 2020-01-01 15:21:00+03:00 — 2026-10-23 20:49:00+03:00
По источникам:
source
Interfax    35525
Name: count, dtype: int64


**Финальные файлы с котировками и новостями:** `data/stocks.csv` и `data/news_final.csv`. Исходные файлы с разбивками по месяцам тоже лежат рядом в папках.